# TPI – Notebook Maestro

## Análisis de cancelaciones de reservas hoteleras

**Materia:** Introducción al Análisis de Datos  
**Comisión:** 10 (Grupo J)  
**Integrante:** Nicolás Viruel  
**Año:** 2026

> Documento integrador del TPI. Secciones 1–4 sincronizadas con Entregas 1 y 2 (Semanas 3–4).

# 1. Definición del problema

El problema consiste en analizar las **cancelaciones de reservas hoteleras** antes del check-in.

Las cancelaciones afectan la planificación de ocupación, disponibilidad de habitaciones e ingresos. El objetivo es identificar qué características de las reservas se relacionan con la cancelación.

**Variable objetivo:** `is_canceled` (0 = no cancelada, 1 = cancelada).

# 2. Preguntas de análisis

1. ¿Qué **porcentaje** de reservas se cancela en el dataset?
2. ¿Las cancelaciones varían según el **tipo de depósito**, **canal de reserva** o **anticipación** (`lead_time`)?
3. ¿Hay diferencias por **temporada**, **tipo de habitación** o **país** del cliente?
4. ¿Clientes **repetidos** cancelan menos que clientes nuevos?
5. ¿Qué variables parecen más relacionadas con `is_canceled` en una primera exploración?
6. ¿Qué decisiones concretas podría tomar el hotel si confirma ciertos patrones?

# 3. Descripción del dataset

- **Archivo:** `hotel_booking_TPI_grupo_J.csv` (Comisión 10, Grupo J)
- **Origen:** subset asignado por la cátedra (Hotel Booking Demand)
- **Unidad:** Entrega 1 – caracterización descriptiva inicial

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('../datos')
archivo = DATA_DIR / 'hotel_booking_TPI_grupo_J.csv'
df_hotel = pd.read_csv(archivo)
print('Dataset cargado:', df_hotel.shape)

Dataset cargado: (25000, 32)


In [2]:
df_hotel.head()

,booking_id,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,arrival_date,stays_in_weekend_nights,...,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests
0,HB-035978,Resort Hotel,0,261,2025,April,18,30,2025-04-30,2,...,A,0,No Deposit,40.0,NaN,0,Contract,42.95,0,1
1,HB-004892,Resort Hotel,1,85,2024,April,14,7,2024-04-07,0,...,A,0,No Deposit,67.0,NaN,0,Transient-Party,64.00,0,0
2,HB-076102,City Hotel,1,364,2023,October,42,16,2023-10-16,0,...,A,0,Non Refund,6.0,NaN,0,Transient-Party,101.50,0,0
3,HB-102752,City Hotel,0,32,2024,December,49,4,2024-12-04,2,...,D,0,No Deposit,9.0,NaN,0,Transient,114.00,0,0
4,HB-069831,City Hotel,1,126,2025,June,23,7,2025-06-07,0,...,A,0,No Deposit,27.0,NaN,0,Transient,89.10,0,1


In [3]:
filas, columnas = df_hotel.shape
print(f'Filas (reservas): {filas}')
print(f'Columnas (variables): {columnas}')
print('\nColumnas:', df_hotel.columns.tolist())

Filas (reservas): 25000
Columnas (variables): 32

Columnas: ['booking_id', 'hotel', 'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'arrival_date', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'agent', 'company', 'days_in_waiting_list', 'customer_type', 'adr', 'required_car_parking_spaces', 'total_of_special_requests']


In [4]:
print(df_hotel.dtypes)
print('\n--- info() ---')
df_hotel.info()

booking_id                            str
hotel                                 str
is_canceled                         int64
lead_time                           int64
arrival_date_year                   int64
arrival_date_month                    str
arrival_date_week_number            int64
arrival_date_day_of_month           int64
arrival_date                          str
stays_in_weekend_nights             int64
stays_in_week_nights                int64
adults                              int64
children                          float64
babies                              int64
meal                                  str
country                               str
market_segment                        str
distribution_channel                  str
is_repeated_guest                   int64
previous_cancellations              int64
previous_bookings_not_canceled      int64
reserved_room_type                    str
assigned_room_type                    str
booking_changes                   

In [5]:
numericas = [
    'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_week_number',
    'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights',
    'adults', 'children', 'babies', 'previous_cancellations',
    'previous_bookings_not_canceled', 'booking_changes', 'days_in_waiting_list',
    'adr', 'required_car_parking_spaces', 'total_of_special_requests',
]
categoricas = [
    'booking_id', 'hotel', 'arrival_date_month', 'arrival_date', 'meal', 'country',
    'market_segment', 'distribution_channel', 'reserved_room_type', 'assigned_room_type',
    'deposit_type', 'customer_type',
]
print('Numéricas:', len(numericas), '| Categóricas:', len(categoricas))

Numéricas: 17 | Categóricas: 12


In [6]:
df_hotel.describe().round(2)

,is_canceled,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,agent,company,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests
count,25000.00,25000.00,25000.00,25000.00,25000.00,25000.00,25000.0,25000.00,25000.00,25000.00,25000.00,25000.00,25000.00,25000.00,21521.00,1440.00,25000.00,25000.00,25000.00,25000.00
mean,0.37,104.22,2024.16,26.69,15.81,0.92,2.5,1.85,0.10,0.01,0.03,0.09,0.14,0.23,86.43,187.33,2.28,101.68,0.06,0.57
std,0.48,107.10,0.71,13.41,8.75,1.00,1.9,0.57,0.39,0.09,0.18,0.87,1.61,0.64,111.05,128.59,17.08,47.97,0.24,0.79
min,0.00,0.00,2023.00,1.00,1.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,9.00,0.00,0.00,0.00,0.00
25%,0.00,18.00,2024.00,16.00,8.00,0.00,1.0,2.00,0.00,0.00,0.00,0.00,0.00,0.00,9.00,62.00,0.00,69.03,0.00,0.00
50%,0.00,69.00,2024.00,27.00,16.00,1.00,2.0,2.00,0.00,0.00,0.00,0.00,0.00,0.00,14.00,186.00,0.00,94.50,0.00,0.00
75%,1.00,160.00,2025.00,37.00,23.00,2.00,3.0,2.00,0.00,0.00,0.00,0.00,0.00,0.00,229.00,268.00,0.00,126.00,0.00,1.00
max,1.00,629.00,2025.00,52.00,31.00,18.00,42.0,40.00,3.00,2.00,1.00,26.00,72.00,18.00,535.00,541.00,391.00,387.00,3.00,5.00


In [7]:
print('Distribución is_canceled:')
print(df_hotel['is_canceled'].value_counts())
print(f"\nPorcentaje cancelaciones: {df_hotel['is_canceled'].mean() * 100:.2f}%")
print('\nTipo de hotel:')
print(df_hotel['hotel'].value_counts())
print('\nCanal de distribución:')
print(df_hotel['distribution_channel'].value_counts().head())

Distribución is_canceled:
is_canceled
0    15741
1     9259
Name: count, dtype: int64

Porcentaje cancelaciones: 37.04%

Tipo de hotel:
hotel
City Hotel      16614
Resort Hotel     8386
Name: count, dtype: int64

Canal de distribución:
distribution_channel
TA/TO        20384
Direct        3140
Corporate     1431
GDS             45
Name: count, dtype: int64


### Observaciones iniciales (Entrega 1)

- **25.000 reservas** × **32 variables**.
- ~**37%** de cancelaciones; nivel relevante para el negocio.
- Predominan reservas en **City Hotel** y canal **TA/TO**.
- `agent` y `company` presentan muchos faltantes (se documentan en la limpieza).

# 4. Preparación y limpieza de datos

**Unidad:** Entrega 2 – diagnóstico y limpieza inicial (Semana 4).

Se conserva el dataset original y se trabaja sobre una copia. Cada decisión queda registrada en la bitácora.

In [8]:
df_hotel_original = df_hotel.copy()
df = df_hotel.copy()

registros_bitacora = []

def registrar(problema, variable, decision, justificacion):
    registros_bitacora.append({
        'problema_detectado': problema,
        'variable': variable,
        'decision': decision,
        'justificacion': justificacion,
    })

In [9]:
print('Dimensiones:', df.shape)
print('Duplicados exactos:', df.duplicated().sum())
cols_sugeridas = [
    'children', 'adults', 'babies', 'stays_in_weekend_nights',
    'stays_in_week_nights', 'adr', 'lead_time', 'country', 'agent', 'company',
]
df[cols_sugeridas].describe(include='all').T

Dimensiones: (25000, 32)


Duplicados exactos: 0


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
children,25000.0,NaN,NaN,NaN,0.10308,0.393218,0.0,0.0,0.0,0.0,3.0
adults,25000.0,NaN,NaN,NaN,1.85444,0.571371,0.0,2.0,2.0,2.0,40.0
babies,25000.0,NaN,NaN,NaN,0.00804,0.091519,0.0,0.0,0.0,0.0,2.0
stays_in_weekend_nights,25000.0,NaN,NaN,NaN,0.92044,0.998193,0.0,0.0,1.0,2.0,18.0
stays_in_week_nights,25000.0,NaN,NaN,NaN,2.499,1.903613,0.0,1.0,2.0,3.0,42.0
adr,25000.0,NaN,NaN,NaN,101.681678,47.969113,0.0,69.0275,94.5,126.0,387.0
lead_time,25000.0,NaN,NaN,NaN,104.22324,107.098048,0.0,18.0,69.0,160.0,629.0
country,24883,134,PRT,10159,NaN,NaN,NaN,NaN,NaN,NaN,NaN
agent,21521.0,NaN,NaN,NaN,86.426746,111.048259,1.0,9.0,14.0,229.0,535.0
company,1440.0,NaN,NaN,NaN,187.325694,128.592996,9.0,62.0,186.0,268.0,541.0


In [10]:
nulos = df.isna().sum().sort_values(ascending=False)
print(nulos[nulos > 0])

registrar('Alta proporción de company nulo', 'company', 'Conservar NaN',
          '94% sin empresa; patrón real, no error masivo.')
registrar('Agent nulo en ~14%', 'agent', 'Conservar NaN',
          'Reservas directas o sin intermediario.')
registrar('Country ausente en 117 reservas', 'country', 'Conservar NaN',
          'Menos del 0,5% del total; eliminar sesgaría países minoritarios.')

company    23560
agent       3479
country      117
dtype: int64


In [11]:
print('adr <= 0:', (df['adr'] <= 0).sum())
print('adults == 0:', (df['adults'] == 0).sum())
print('Sin huéspedes:', ((df['adults'] + df['children'] + df['babies']) == 0).sum())

registrar('adr nulo, cero o negativo', 'adr', 'Marcar; conservar por ahora',
          'Puede ser cortesía o error; no eliminar sin contexto de negocio.')
registrar('adults == 0', 'adults', 'Conservar y analizar',
          'Posible error o reserva especial.')
if ((df['adults'] + df['children'] + df['babies']) == 0).any():
    registrar('Estadía sin huéspedes', 'adults, children, babies',
              'Conservar para revisión', 'Combinación atípica en operación normal.')

adr <= 0: 446
adults == 0: 80
Sin huéspedes: 37


In [12]:
def detectar_outliers_iqr(serie):
    s = serie.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return s[(s < lim_inf) | (s > lim_sup)]

for col in ['lead_time', 'adr', 'stays_in_week_nights', 'stays_in_weekend_nights']:
    out = detectar_outliers_iqr(df[col])
    print(f'{col}: {len(out)} atípicos IQR')

registrar('lead_time muy alto', 'lead_time', 'Conservar',
          'Anticipación extrema posible en resorts.')
registrar('adr atípico por IQR', 'adr', 'Conservar',
          'Tarifas premium válidas; evaluar en EDA.')

lead_time: 657 atípicos IQR
adr: 777 atípicos IQR
stays_in_week_nights: 679 atípicos IQR
stays_in_weekend_nights: 52 atípicos IQR


In [13]:
df['arrival_date'] = pd.to_datetime(df['arrival_date'], errors='coerce')
print('Fechas inválidas:', df['arrival_date'].isna().sum())
print('Rango:', df['arrival_date'].min(), '→', df['arrival_date'].max())
registrar('Unificación arrival_date', 'arrival_date', 'Convertir a datetime',
          'Coherencia temporal; sin fechas inválidas detectadas.')

Fechas inválidas:

 0
Rango: 2023-07-01 00:00:00 → 2025-08-31 00:00:00


In [14]:
bitacora = pd.DataFrame(registros_bitacora)
bitacora

,problema_detectado,variable,decision,justificacion
0,Alta proporción de company nulo,company,Conservar NaN,"94% sin empresa; patrón real, no error masivo."
1,Agent nulo en ~14%,agent,Conservar NaN,Reservas directas o sin intermediario.
2,Country ausente en 117 reservas,country,Conservar NaN,"Menos del 0,5% del total; eliminar sesgaría pa..."
3,"adr nulo, cero o negativo",adr,Marcar; conservar por ahora,Puede ser cortesía o error; no eliminar sin co...
4,adults == 0,adults,Conservar y analizar,Posible error o reserva especial.
5,Estadía sin huéspedes,"adults, children, babies",Conservar para revisión,Combinación atípica en operación normal.
6,lead_time muy alto,lead_time,Conservar,Anticipación extrema posible en resorts.
7,adr atípico por IQR,adr,Conservar,Tarifas premium válidas; evaluar en EDA.
8,Unificación arrival_date,arrival_date,Convertir a datetime,Coherencia temporal; sin fechas inválidas dete...


In [15]:
df_hotel_limpio = df.copy()
print('Original:', df_hotel_original.shape, '| Limpio:', df_hotel_limpio.shape)
df_hotel_limpio[cols_sugeridas].head()

Original: (25000, 32) | Limpio: (25000, 32)


,children,adults,babies,stays_in_weekend_nights,stays_in_week_nights,adr,lead_time,country,agent,company
0,0.0,2,0,2,5,42.95,261,GBR,40.0,NaN
1,0.0,2,0,0,3,64.00,85,PRT,67.0,NaN
2,0.0,2,0,0,2,101.50,364,PRT,6.0,NaN
3,0.0,1,0,2,0,114.00,32,CHE,9.0,NaN
4,0.0,2,0,0,2,89.10,126,PRT,27.0,NaN


### Cierre limpieza (Semana 4)

Diagnóstico inicial completado: faltantes documentados, valores imposibles identificados y atípicos analizados con IQR **sin eliminación automática**. Transformaciones adicionales se completarán en Semana 5 (Entrega 2 formal).

# 5. Análisis exploratorio de datos

*Pendiente – Semana 5 en adelante.*

Visualizaciones, tablas cruzadas y comparaciones por `hotel`, `deposit_type`, `lead_time`, `country`, etc.

# 6. Análisis de la variable objetivo: is_canceled

La variable objetivo es **`is_canceled`**:

- `0`: reserva no cancelada
- `1`: reserva cancelada

En la caracterización inicial (Sección 3) se observó ~37% de cancelaciones. En entregas futuras se profundizará qué variables se asocian con cada valor.

In [16]:
pct_cancel = df_hotel['is_canceled'].mean() * 100
print(f'Cancelaciones: {pct_cancel:.2f}%')
df_hotel.groupby('hotel')['is_canceled'].mean().mul(100).round(2)

Cancelaciones: 37.04%


hotel
City Hotel      41.72
Resort Hotel    27.76
Name: is_canceled, dtype: float64

# 7. Modelado

*Pendiente – unidades posteriores.*

Modelo de clasificación, partición train/test, métricas (accuracy, precision, recall, F1).

# 8. Resultados y conclusiones

*Pendiente – entrega final.*

Conclusiones vinculadas a las preguntas de la Sección 2, limitaciones y recomendaciones al hotel.